# Kinetiscope Kinetic Modeling

This notebook demonstrates how to create pulse profiles for Kinetiscope simulations and analyze the results.

## Workflow Overview

1. Create pump-probe pulse sequences
2. Run Kinetiscope simulation (external)
3. Load and analyze results
4. Calculate differential signals
5. Compare with experimental data

In [ ]:
# Configuration
from pathlib import Path

# Output directory for pulse profiles
PROFILE_DIR = Path("../kinetiscope_profiles/demo")

# Pulse parameters
PUMP_FWHM = 40e-15   # 40 fs
PROBE_FWHM = 80e-15  # 80 fs

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from dssc.kinetiscope import (
    create_pump_probe_sequence,
    write_prf_file,
    create_delay_series,
    standard_delay_series,
    gaussian_pulse,
)

## Part 1: Creating Pulse Profiles

Generate Gaussian pump and probe pulses with specified delays.

In [ ]:
# Create a single pump-probe sequence
delay = 1e-12  # 1 ps delay

pump_probe, probe_only = create_pump_probe_sequence(
    delay=delay,
    pump_fwhm=PUMP_FWHM,
    probe_fwhm=PROBE_FWHM,
    pump_amplitude=1.0,
    probe_amplitude=0.1,
)

print(f"Pump-probe profile: {len(pump_probe.time)} points")
print(f"Time range: {pump_probe.time[0]*1e15:.0f} fs - {pump_probe.time[-1]*1e12:.0f} ps")

In [ ]:
# Visualize the pulse sequence
fig, ax = plt.subplots(figsize=(10, 5))

# Convert to ps for plotting
time_ps = pump_probe.time * 1e12

ax.plot(time_ps, pump_probe.intensity, 'k-', label='Combined', linewidth=2)
ax.plot(time_ps, probe_only.intensity, 'r--', label='Probe only', linewidth=1.5)

ax.set_xlabel('Time (ps)')
ax.set_ylabel('Intensity (a.u.)')
ax.set_title(f'Pump-Probe Pulse Sequence (delay = {delay*1e12:.0f} ps)')
ax.legend()
ax.set_xlim(0, 2)
plt.show()

### Generate Full Delay Series

Create profiles for delays spanning femtoseconds to microseconds.

In [ ]:
# Standard delay series
delays = standard_delay_series()

print(f"Number of delays: {len(delays)}")
print(f"\nSample delays:")
print(f"  Negative: {delays[delays < 0] * 1e15} fs")
print(f"  fs range: {delays[(delays > 0) & (delays < 1e-12)] * 1e15} fs")
print(f"  ps range: {delays[(delays >= 1e-12) & (delays < 1e-9)] * 1e12} ps")
print(f"  ns range: {delays[delays >= 1e-9] * 1e9} ns")

In [ ]:
# Create all profiles (uncomment to write files)
# create_delay_series(
#     delays,
#     PROFILE_DIR,
#     pump_fwhm=PUMP_FWHM,
#     probe_fwhm=PROBE_FWHM,
# )
# print(f"Profiles written to: {PROFILE_DIR}")

## Part 2: Running Kinetiscope

**Kinetiscope is external software.** After creating pulse profiles:

1. Open Kinetiscope
2. Load your reaction mechanism (.rxn file)
3. Set the pulse profile (.prf file)
4. Run simulation
5. Export results

Repeat for each delay time with pump-on and pump-off conditions.

## Part 3: Analyzing Results

Load Kinetiscope output and calculate differential signals.

In [ ]:
from dssc.kinetiscope import (
    read_kinetiscope_result,
    calculate_differential_signal,
    KinetiscopeResult,
)

In [ ]:
# For demo, create synthetic Kinetiscope-like results
# In real use: result = read_kinetiscope_result("path/to/result.txt")

time_sim = np.logspace(-15, -6, 200)  # fs to μs

# Simulate excited state dynamics
# S1 -> T1 intersystem crossing (100 ps)
# T1 -> S0 decay (1 μs)
s1 = np.exp(-time_sim / 100e-12)
t1 = (1 - np.exp(-time_sim / 100e-12)) * np.exp(-time_sim / 1e-6)
s0 = 1 - s1 - t1

# Spectral signals
gsb = -s1 - t1  # Both excited states deplete ground
esa = s1 * 0.5 + t1 * 0.3  # S1 and T1 have different absorption
ems = s1 * 0.2  # Only S1 emits

pump_on = KinetiscopeResult(
    time=time_sim, s0=s0, s1=s1, t1=t1,
    gsb=gsb, esa=esa, ems=ems
)

# Pump-off: no excitation
pump_off = KinetiscopeResult(
    time=time_sim,
    s0=np.ones_like(time_sim),
    s1=np.zeros_like(time_sim),
    t1=np.zeros_like(time_sim),
    gsb=np.zeros_like(time_sim),
    esa=np.zeros_like(time_sim),
    ems=np.zeros_like(time_sim),
)

In [ ]:
# Calculate differential signals
diff_signals = calculate_differential_signal(pump_on, pump_off)

print("Differential signals calculated:")
for key in diff_signals:
    print(f"  {key}: max = {np.max(np.abs(diff_signals[key])):.3f}")

### Plot Population Dynamics

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Population dynamics
ax = axes[0]
ax.semilogx(time_sim * 1e12, pump_on.s0, 'b-', label='S₀', linewidth=2)
ax.semilogx(time_sim * 1e12, pump_on.s1, 'r-', label='S₁', linewidth=2)
ax.semilogx(time_sim * 1e12, pump_on.t1, 'g-', label='T₁', linewidth=2)
ax.set_xlabel('Time (ps)')
ax.set_ylabel('Population')
ax.set_title('Excited State Population Dynamics')
ax.legend()
ax.set_xlim(1e-3, 1e6)

# Spectral signals
ax = axes[1]
ax.semilogx(time_sim * 1e12, diff_signals['gsb'], 'b-', label='GSB', linewidth=2)
ax.semilogx(time_sim * 1e12, diff_signals['esa'], 'r-', label='ESA', linewidth=2)
ax.semilogx(time_sim * 1e12, diff_signals['ems'], 'g-', label='Emission', linewidth=2)
ax.axhline(0, color='k', linestyle='--', linewidth=0.5)
ax.set_xlabel('Time (ps)')
ax.set_ylabel('ΔSignal')
ax.set_title('Differential Spectral Signals')
ax.legend()
ax.set_xlim(1e-3, 1e6)

plt.tight_layout()
plt.show()

## Summary

1. **Create pulse profiles** - Gaussian pump + probe with specified delays
2. **Run Kinetiscope** - External simulation with reaction mechanism
3. **Analyze results** - Extract populations and spectral features
4. **Calculate differentials** - Pump-on minus pump-off
5. **Compare with experiment** - Validate kinetic model

### Key Parameters

- Pump FWHM: typically 30-50 fs
- Probe FWHM: typically 50-100 fs
- Delay range: -500 fs to 1 μs (or longer)
- Time resolution in simulation: 0.5 fs